In [2]:
!optimum-cli export onnx --model ./finetuned_whisper_model_depth_6/saved_model --library-name transformers --task automatic-speech-recognition-with-past ./whisper_depth_6_onnx

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'suppress_tokens': []}
C:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\transformers\models\whisper\modeling_whisper.py:1159: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if input_features.shape[-1] != expected_seq_length:
C:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\transformers\models\whisper\modeling_whisper.py:338: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flo

In [5]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq

MODEL_NAME = "./finetuned_whisper_model_depth_6/saved_processor"  # or your depth-reduced checkpoint path
EXPORT_PATH = "./whisper_depth_6_onnx"


processor = WhisperProcessor.from_pretrained(MODEL_NAME)
processor.save_pretrained(EXPORT_PATH)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


[]

In [3]:
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from transformers import WhisperProcessor

model = ORTModelForSpeechSeq2Seq.from_pretrained("./whisper_depth_6_onnx")
processor = WhisperProcessor.from_pretrained("./whisper_depth_6_onnx")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [17]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

QUANT_PATH = "./whisper_depth_6_onnx_int8"

qconfig = AutoQuantizationConfig.arm64(
    is_static=False,
    per_channel=True
)

for component in [
    "encoder_model",
    "decoder_model",
    "decoder_with_past_model"
]:
    quantizer = ORTQuantizer.from_pretrained(
        EXPORT_PATH,
        file_name=f"{component}.onnx"
    )

    quantizer.quantize(
        save_dir=QUANT_PATH,
        quantization_config=qconfig
    )

processor.save_pretrained(QUANT_PATH)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'suppress_tokens': []}
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-defau

[]

In [18]:
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq

quantized_model = ORTModelForSpeechSeq2Seq.from_pretrained(
    QUANT_PATH,
    encoder_file_name="encoder_model_quantized.onnx",
    decoder_file_name="decoder_model_quantized.onnx",
    decoder_with_past_file_name="decoder_with_past_model_quantized.onnx",
)

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset(
    "Kennethdot/Ghana_English-Twi_Code-switching_Speech",
    )

dataset = dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)


In [9]:
def transcribe_from_dataset(dataset_sample, whisper_model, max_new_tokens=128):
    input_features = processor.feature_extractor(
        dataset_sample["array"],
        sampling_rate=dataset_sample["sampling_rate"],
        return_tensors="pt"
    ).input_features
    # no .to(model.device) — ORTModel handles device/session placement internally

    predicted_ids = whisper_model.generate(
        input_features,
        max_new_tokens=max_new_tokens,
        task="transcribe",
        forced_decoder_ids=None
    )
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    return transcription[0].strip()

In [10]:
import re

def normalize_cs(text):
    text = text.lower()

    # keep English letters + Twi chars
    text = re.sub(r"[^a-z0-9ɔɛ\s']", "", text)

    # normalize apostrophes (optional)
    text = re.sub(r"'", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [11]:
from evaluate import load as metrics_loader
wer_metric = metrics_loader("wer")

def get_wer(references, predictions, normalize=True, verbose=True):
  rs = references
  ps = predictions
  if normalize:
    ps = [normalize_cs(x) for x in predictions]
    rs = [normalize_cs(x) for x in references]
  if verbose:
    for r, p in zip(rs, ps):
      print(r)
      print(p)
      print()

  return wer_metric.compute(references=rs, predictions=ps)

In [ ]:
from transformers import GenerationConfig

fresh_gen_config = GenerationConfig.from_pretrained("kennethdot/kasanoma_whisper")
model.generation_config = fresh_gen_config
model.generation_config.save_pretrained(QUANT_PATH)

In [13]:
normalize_for_wer_calc = True #@param{type: 'boolean'}

print('number of test examples to process:', len(dataset['test']))

predictions = []
finetuned_predictions = []
references  = []

for idx in range(len(dataset['test'])):
  print('inference on example:', idx)
  sample = dataset['test'][idx]["audio"]
  predictions.append(transcribe_from_dataset(sample, quantized_model))
  references.append(dataset['test'][idx]['transcript'])

number of test examples to process: 1731
inference on example: 0
inference on example: 1
inference on example: 2
inference on example: 3
inference on example: 4
inference on example: 5
inference on example: 6
inference on example: 7
inference on example: 8
inference on example: 9
inference on example: 10
inference on example: 11
inference on example: 12
inference on example: 13
inference on example: 14
inference on example: 15
inference on example: 16
inference on example: 17
inference on example: 18
inference on example: 19
inference on example: 20
inference on example: 21
inference on example: 22
inference on example: 23
inference on example: 24
inference on example: 25
inference on example: 26
inference on example: 27
inference on example: 28
inference on example: 29
inference on example: 30
inference on example: 31
inference on example: 32
inference on example: 33
inference on example: 34
inference on example: 35
inference on example: 36
inference on example: 37
inference on exampl

KeyboardInterrupt: 

In [14]:
finetuned_wer = get_wer(references=references, predictions=predictions, normalize=normalize_for_wer_calc, verbose=False)
from jiwer import wer, cer

overall_wer = wer(references, predictions)
overall_cer = cer(references, predictions)

print(f"Overall WER: {overall_wer:.3f}")
print(f"Overall CER: {overall_cer:.3f}")
print(f'FINETUNED WER: {finetuned_wer}')

Overall WER: 0.268
Overall CER: 0.207
FINETUNED WER: 0.25159982935153585


In [19]:
import os

def get_onnx_model_size_mb(model_path):
    """Sum the size of all .onnx files in the model directory."""
    total_bytes = 0
    for fname in os.listdir(model_path):
        if fname.endswith(".onnx"):
            fpath = os.path.join(model_path, fname)
            total_bytes += os.path.getsize(fpath)
    return total_bytes / (1024 * 1024)

quantized_size_mb = get_onnx_model_size_mb(QUANT_PATH)
print(f"Quantized model size: {quantized_size_mb:.2f} MB")

Quantized model size: 229.10 MB


In [16]:
unquantized_size_mb = get_onnx_model_size_mb(EXPORT_PATH)
print(f"Unquantized ONNX model size: {unquantized_size_mb:.2f} MB")
print(f"Compression ratio: {unquantized_size_mb / quantized_size_mb:.2f}x")

Unquantized ONNX model size: 1256.66 MB
Compression ratio: 5.49x


In [20]:
finetuned_wer = get_wer(references=references, predictions=predictions, normalize=normalize_for_wer_calc, verbose=False)

print(f'FINETUNED WER: {finetuned_wer}')

FINETUNED WER: 0.25159982935153585


In [21]:
for transcript, prediction in zip(references, predictions):
  print('REFERENCE:', transcript)
  print('PREDICTION:', prediction)
  print()

REFERENCE: “Sɛ wopɛ a, yɛbɛtumi anante, it’s not that far.”
PREDICTION: “Sɛ wopɛ a, yɛbɛtumi anante, it’s not that paa.”

REFERENCE: Ɛyɛ den oo, but I’ll try.
PREDICTION: Ɛyɛ den oo, but I’ll try.

REFERENCE: Cough syrup no yɛ dɛ dodo.
PREDICTION: Cough syrup no yɛ dɛbɛ bi.

REFERENCE: Nyame ne hene daa, God is good.
PREDICTION: Nyame ne hene daa, God is good ayɛ.

REFERENCE: Mepakyew, boa me, it is an emergency.
PREDICTION: Mepakyew, boa me, it is anɛ, ɛbɛ nsuo.

REFERENCE: “Wo ho te sɛn? You’ve been quiet all day.”
PREDICTION: “Wo ho te sɛn? You’ve been quiet all day.”

REFERENCE: Ma yɛnkɔ y’anim kakra, let’s skip this place.
PREDICTION: Ma yɛnkɔ y’anim kakra, let’s skip this place.

REFERENCE: She wore a white cloth because na n’ani agye.
PREDICTION: She wore a white cloth because na n’ani agye.

REFERENCE: Menni cash, can I pay with my phone?
PREDICTION: Menni cash, can I pay with my phone?

REFERENCE: Anka mepɛ waakye, but it is finished.
PREDICTION: Anka mepɛ waakye, but it is fi

# Whisper.cpp Quantization

In [22]:
!git clone https://github.com/openai/whisper openai-whisper-repo

fatal: destination path 'openai-whisper-repo' already exists and is not an empty directory.


In [24]:
!python whisper.cpp/models/convert-h5-to-ggml.py ./finetuned_whisper_model_depth_6 ./openai-whisper-repo ./whisper.cpp/models

model.encoder.conv1.weight  ->  encoder.conv1.weight
encoder.conv1.weight 3 (768, 80, 3)
model.encoder.conv1.bias  ->  encoder.conv1.bias
  Reshaped variable:  encoder.conv1.bias  to shape:  (768, 1)
encoder.conv1.bias 2 (768, 1)
  Converting to float32
model.encoder.conv2.weight  ->  encoder.conv2.weight
encoder.conv2.weight 3 (768, 768, 3)
model.encoder.conv2.bias  ->  encoder.conv2.bias
  Reshaped variable:  encoder.conv2.bias  to shape:  (768, 1)
encoder.conv2.bias 2 (768, 1)
  Converting to float32
model.encoder.embed_positions.weight  ->  encoder.positional_embedding
encoder.positional_embedding 2 (1500, 768)
  Converting to float32
model.encoder.layers.0.self_attn.k_proj.weight  ->  encoder.blocks.0.attn.key.weight
encoder.blocks.0.attn.key.weight 2 (768, 768)
model.encoder.layers.0.self_attn.v_proj.weight  ->  encoder.blocks.0.attn.value.weight
encoder.blocks.0.attn.value.weight 2 (768, 768)
model.encoder.layers.0.self_attn.v_proj.bias  ->  encoder.blocks.0.attn.value.bias
enco

In [29]:
import os
print(os.listdir("./whisper.cpp/models"))

['.gitignore', 'convert-h5-to-coreml.py', 'convert-h5-to-ggml.py', 'convert-parakeet-to-ggml.py', 'convert-pt-to-ggml.py', 'convert-silero-vad-to-ggml.py', 'convert-whisper-to-coreml.py', 'convert-whisper-to-openvino.py', 'download-coreml-model.sh', 'download-ggml-model.cmd', 'download-ggml-model.sh', 'download-vad-model.cmd', 'download-vad-model.sh', 'for-tests-ggml-base.bin', 'for-tests-ggml-base.en.bin', 'for-tests-ggml-large.bin', 'for-tests-ggml-medium.bin', 'for-tests-ggml-medium.en.bin', 'for-tests-ggml-parakeet-tdt-bad-nfft0.bin', 'for-tests-ggml-parakeet-tdt.bin', 'for-tests-ggml-small.bin', 'for-tests-ggml-small.en.bin', 'for-tests-ggml-tiny.bin', 'for-tests-ggml-tiny.en.bin', 'for-tests-silero-v6.2.0-ggml.bin', 'generate-coreml-interface.sh', 'generate-coreml-model.sh', 'generate-parakeet-test-model.py', 'ggml-model.bin', 'ggml_to_pt.py', 'README.md', 'requirements-coreml.txt', 'requirements-openvino.txt', 'requirements-parakeet.txt']


In [30]:
import shutil
os.mkdir("./model_export_depth_6")
shutil.copy2("./whisper.cpp/models/ggml-model.bin", "./model_export_depth_6/ggml-model.bin")

'./model_export_depth_6/ggml-model.bin'

In [32]:
size_mb = os.path.getsize("./model_export_depth_6/ggml-model.bin") / (1024 * 1024)
print(f"GGML model size: {size_mb:.2f} MB")

GGML model size: 275.49 MB


In [42]:
!winget install Kitware.CMake

Found CMake [Kitware.CMake] Version 4.4.2
This application is licensed to you by its owner.
Microsoft is not responsible for, nor does it grant any licenses to, third-party packages.
Successfully verified installer hash
Starting package install...


In [5]:
!cmake --version

cmake version 4.4.2

CMake suite maintained and supported by Kitware (kitware.com/cmake).


In [26]:
!cmake -S whisper.cpp -B whisper.cpp/build -DCMAKE_BUILD_TYPE=Release
!cmake --build whisper.cpp/build --config Release

-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: AMD64
-- CMAKE_GENERATOR_PLATFORM: 
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: /arch:AVX2 GGML_AVX2;GGML_FMA;GGML_F16C
-- ggml version: 0.19.0
-- ggml commit:  592feef0
-- Configuring done (1.1s)
-- Generating done (1.9s)
-- Build files have been written to: C:/Users/KASA/Downloads/Kasa/quant/whisper.cpp/build


CMake Warning (deprecated) at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.
This warning is for project developers.  Use -Wno-author or -Wno-deprecated
to suppress it.



MSBuild version 18.8.2+ce25c0108 for .NET Framework

  1>Checking Build System
  bench.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\bench.exe
  ggml-base.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\ggml-base.dll
  ggml-cpu.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\ggml-cpu.dll
  ggml.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\ggml.dll
  Auto build dll exports
  whisper.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\whisper.dll
  common.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\examples\Release\common.lib
  main.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\main.exe
  Auto build dll exports
  parakeet.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\parakeet.dll
  parakeet-cli.vcxproj -> C:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\p

In [33]:
for root, dirs, files in os.walk("whisper.cpp/build"):
    for f in files:
        if "quantize" in f.lower():
            print(os.path.join(root, f))

whisper.cpp/build\bin\Release\parakeet-quantize.exe
whisper.cpp/build\bin\Release\whisper-quantize.exe
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.vcxproj
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.vcxproj.filters
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.dir\Release\parakeet-quantize.exe.recipe
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.dir\Release\parakeet-quantize.obj
whisper.cpp/build\examples\parakeet-quantize\parakeet-quantize.dir\Release\parakeet.AF828641.tlog\parakeet-quantize.lastbuildstate
whisper.cpp/build\examples\quantize\whisper-quantize.vcxproj
whisper.cpp/build\examples\quantize\whisper-quantize.vcxproj.filters
whisper.cpp/build\examples\quantize\whisper-quantize.dir\Release\quantize.obj
whisper.cpp/build\examples\quantize\whisper-quantize.dir\Release\whisper-quantize.exe.recipe
whisper.cpp/build\examples\quantize\whisper-quantize.dir\Release\whisper-quantize.tlog\whisper-quantize.lastbuilds

In [35]:
import os

cwd = os.getcwd()
quantize_bin = os.path.join(cwd, "whisper.cpp", "build", "bin", "Release", "whisper-quantize.exe")
input_model = os.path.join(cwd, "model_export_depth_6", "ggml-model.bin")
output_model = os.path.join(cwd, "model_export_depth_6", "ggml-model_q_8.bin")

print(quantize_bin)
print(os.path.exists(quantize_bin))

c:\Users\KASA\Downloads\Kasa\quant\whisper.cpp\build\bin\Release\whisper-quantize.exe
True


In [36]:
import subprocess

result = subprocess.run(
    [quantize_bin, input_model, output_model, "q8_0"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

whisper_model_quantize: loading model from 'c:\Users\KASA\Downloads\Kasa\quant\model_export_depth_6\ggml-model.bin'
                                            encoder.conv1.weight - [    3,    80,   768], type =    f16 size =    0.352 MB
                                              encoder.conv1.bias - [    1,   768,     1], type =    f32 size =    0.003 MB
                                            encoder.conv2.weight - [    3,   768,   768], type =    f16 size =    3.375 MB
                                              encoder.conv2.bias - [    1,   768,     1], type =    f32 size =    0.003 MB
                                    encoder.positional_embedding - [  768,  1500,     1], type =    f32 size =    4.395 MB
                                encoder.blocks.0.attn.key.weight - [  768,   768,     1], type =    f16 size =     2.25 MB ->     0.60 MB
                              encoder.blocks.0.attn.value.weight - [  768,   768,     1], type =    f16 size =     2.25 MB ->     0

In [37]:
dataset['test'][0]

{'speaker_id': '35b8',
 'age_range': '18-24',
 'gender': 'male',
 'prompt_set': 'standard',
 'transcript': '“Sɛ wopɛ a, yɛbɛtumi anante, it’s not that far.”',
 'duration': 4.853499889373779,
 'split': 'test',
 'audio': {'path': '35b8_std_1_20260225T213853Z.ogg',
  'array': array([ 2.85863644e-06, -3.76813114e-06, -1.07599772e-05, ...,
          1.29173987e-03,  1.86404213e-04,  3.27933254e-03]),
  'sampling_rate': 16000},
 'file_name': None,
 'error': None}

In [38]:
import soundfile as sf
import json
references = {}
os.makedirs("wavs", exist_ok=True)

for i, sample in enumerate(dataset["test"]):
    audio = sample["audio"]
    # use the audio path as a stable id, falling back to index
    file_id = os.path.splitext(os.path.basename(audio["path"]))[0] if audio["path"] else f"{i:05d}"
    sf.write(f"wavs/{file_id}.wav", audio["array"], audio["sampling_rate"])
    references[file_id] = sample["transcript"]

with open("references.json", "w", encoding="utf-8") as f:
    json.dump(references, f, ensure_ascii=False, indent=2)

In [42]:
import subprocess
import os

binary_path = r"whisper.cpp\build\bin\Release\whisper-cli.exe"
model_path = "./model_export_depth_6/ggml-model_q_8.bin"

print("binary exists:", os.path.exists(binary_path))
print("model exists:", os.path.exists(model_path))

binary exists: True
model exists: True


In [43]:
f = "wavs/" + os.listdir("wavs")[0]
file_id = os.path.splitext(os.path.basename(f))[0]

os.makedirs("transcripts", exist_ok=True)

result = subprocess.run([
    binary_path,
    "-m", model_path,
    "-f", f,
    "-otxt",
    "-of", f"transcripts/{file_id}"
], capture_output=True, text=True)

print("stdout:", result.stdout)
print("stderr:", result.stderr)
print("return code:", result.returncode)

stdout: 
[00:00:00.960 --> 00:00:30.000]   yÉ› sure? Lunch will end soon o.

stderr: whisper_init_from_file_with_params_no_state: loading model from './model_export_depth_6/ggml-model_q_8.bin'
whisper_init_with_params_no_state: use gpu    = 1
whisper_init_with_params_no_state: flash attn = 1
whisper_init_with_params_no_state: gpu_device = 0
whisper_init_with_params_no_state: dtw        = 0
whisper_init_with_params_no_state: devices    = 1
whisper_init_with_params_no_state: backends   = 1
whisper_model_load: loading model
whisper_model_load: n_vocab       = 51865
whisper_model_load: n_audio_ctx   = 1500
whisper_model_load: n_audio_state = 768
whisper_model_load: n_audio_head  = 12
whisper_model_load: n_audio_layer = 6
whisper_model_load: n_text_ctx    = 448
whisper_model_load: n_text_state  = 768
whisper_model_load: n_text_head   = 12
whisper_model_load: n_text_layer  = 6
whisper_model_load: n_mels        = 80
whisper_model_load: ftype         = 7
whisper_model_load: qntvr         = 2
w

In [46]:
import subprocess
import os
import time
import json


wav_files = [f for f in os.listdir("wavs") if f.endswith(".wav")]
print(f"Total files: {len(wav_files)}")

results = {}
failed = []
start = time.time()

if os.path.exists("transcripts.json"):
    with open("transcripts.json", encoding="utf-8") as f:
        results = json.load(f)
    print(f"Loaded {len(results)} existing transcripts, will skip those")
else:
    results = {}

for i, fname in enumerate(wav_files):
    file_id = os.path.splitext(fname)[0]
    if file_id in results:
        continue  
    result = subprocess.run([
        binary_path,
        "-m", model_path,
        "-f", f"wavs/{fname}",
        "-nt"  # no timestamps, just plain text to stdout
    ], capture_output=True, text=True, encoding="utf-8", errors="replace")

    if result.returncode != 0:
        failed.append((fname, result.stderr))
        continue

    results[file_id] = result.stdout.strip()

    if (i + 1) % 20 == 0:
        elapsed = time.time() - start
        print(f"{i+1}/{len(wav_files)} done ({elapsed:.1f}s elapsed)")
        # save progress incrementally so a crash doesn't lose everything
        with open("transcripts.json", "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

# final save
with open("transcripts.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nFinished. {len(results)} succeeded, {len(failed)} failed.")
for fname, err in failed[:5]:
    print(fname, "->", err[:200])

Total files: 1651
20/1651 done (28.9s elapsed)
40/1651 done (59.8s elapsed)
60/1651 done (90.3s elapsed)
80/1651 done (121.1s elapsed)
100/1651 done (151.4s elapsed)
120/1651 done (181.8s elapsed)
140/1651 done (212.5s elapsed)
160/1651 done (243.2s elapsed)
180/1651 done (275.0s elapsed)
200/1651 done (312.2s elapsed)
220/1651 done (351.3s elapsed)
240/1651 done (385.3s elapsed)
260/1651 done (425.4s elapsed)
280/1651 done (470.2s elapsed)
300/1651 done (500.5s elapsed)
320/1651 done (530.6s elapsed)
340/1651 done (560.3s elapsed)
360/1651 done (590.1s elapsed)
380/1651 done (620.0s elapsed)
400/1651 done (650.2s elapsed)
420/1651 done (680.2s elapsed)
440/1651 done (709.8s elapsed)
460/1651 done (739.7s elapsed)
480/1651 done (770.9s elapsed)
500/1651 done (803.1s elapsed)
520/1651 done (833.9s elapsed)
540/1651 done (864.8s elapsed)
560/1651 done (895.6s elapsed)
580/1651 done (925.5s elapsed)
600/1651 done (956.0s elapsed)
620/1651 done (986.7s elapsed)
640/1651 done (1016.8s elaps

In [47]:
import json
from jiwer import wer, cer

# load references
with open("references.json", encoding="utf-8") as f:
    references = json.load(f)

# load transcripts (single-file version from the updated inference loop)
with open("transcripts.json", encoding="utf-8") as f:
    transcripts = json.load(f)

# only score files that succeeded in both dicts
common_ids = [fid for fid in references if fid in transcripts]
missing = [fid for fid in references if fid not in transcripts]

print(f"Scoring {len(common_ids)} / {len(references)} files")
if missing:
    print(f"Missing {len(missing)} transcripts (failed inference?): {missing[:5]}")

refs = [references[fid] for fid in common_ids]
hyps = [transcripts[fid] for fid in common_ids]

overall_wer = wer(refs, hyps)
overall_cer = cer(refs, hyps)

print(f"Overall WER: {overall_wer:.3f}")
print(f"Overall CER: {overall_cer:.3f}")

Scoring 1651 / 1651 files
Overall WER: 0.132
Overall CER: 0.094
